In [ ]:
!pip install -q mne scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 50.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
# CELL 1 (DIRECT DOWNLOAD): Fetch BCIC-IV-2a Subject 1 files from BNCI mirror

import os
import urllib.request
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'
os.makedirs(CROSS_PROJECT_DIR, exist_ok=True)
print('CROSS_PROJECT_DIR:', CROSS_PROJECT_DIR)

# BNCI Horizon 2020 mirror hosts the BCIC-IV-2a dataset files.
BASE_URL = 'http://bnci-horizon-2020.eu/database/data-sets/001-2014/'
FILES = ['A01T.mat', 'A01E.mat']

for fname in FILES:
    url = BASE_URL + fname
    dst = os.path.join(CROSS_PROJECT_DIR, fname)
    if os.path.exists(dst):
        print('Already exists:', dst)
        continue
    print('Downloading {} -> {}'.format(url, dst))
    try:
        urllib.request.urlretrieve(url, dst)
        print('  Saved. Size:', os.path.getsize(dst), 'bytes')
    except Exception as exc:
        print('  Download failed:', exc)
        raise

print('Files present:', sorted([f for f in os.listdir(CROSS_PROJECT_DIR) if f.endswith('.mat')]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CROSS_PROJECT_DIR: /content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/
  Saved. Size: 42806453 bytes
  Saved. Size: 43772146 bytes
Files present: ['A01E.mat', 'A01T.mat']


In [ ]:
# CELL 1B (LOADER): Load BCIC-IV-2a .mat files, filter, split, save

import os
import numpy as np
import scipy.io
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'

mat_path_T = os.path.join(CROSS_PROJECT_DIR, 'A01T.mat')
mat_path_E = os.path.join(CROSS_PROJECT_DIR, 'A01E.mat')
assert os.path.exists(mat_path_T), 'Missing: ' + mat_path_T
assert os.path.exists(mat_path_E), 'Missing: ' + mat_path_E

CH_NAMES = ['Fz','FC3','FC1','FCz','FC2','FC4','C5','C3','C1','Cz','C2','C4','C6',
            'CP3','CP1','CPz','CP2','CP4','P1','Pz','P2','POz']
COMMON_CHANNELS = ['FC3','FC1','FCz','FC2','FC4','C5','C3','C1','Cz','C2','C4','C6',
                   'CP3','CP1','CPz','CP2','CP4']
ch_idx = [CH_NAMES.index(c) for c in COMMON_CHANNELS]
print('Using {} channels: {}'.format(len(COMMON_CHANNELS), COMMON_CHANNELS))

LABEL_MAP = {1: 'left_hand', 2: 'right_hand', 3: 'feet', 4: 'tongue'}

def load_mat_runs(path):
    mat = scipy.io.loadmat(path, squeeze_me=True, struct_as_record=False)
    data = mat['data']
    all_trials, all_labels = [], []
    sfreq = None
    for run in data:
        X = run.X
        trials = run.trial
        y = run.y
        if sfreq is None:
            sfreq = float(run.fs)
        epoch_len = int(sfreq * 4.0)
        for onset, lbl in zip(trials, y):
            start = int(onset)
            end = start + epoch_len
            if end > X.shape[0]:
                continue
            epoch = X[start:end, ch_idx].T
            all_trials.append(epoch)
            all_labels.append(LABEL_MAP[int(lbl)])
    return np.stack(all_trials).astype(np.float32), np.array(all_labels), sfreq

print('Loading A01T (training session)...')
X_T, y_T, sfreq = load_mat_runs(mat_path_T)
print('  ', X_T.shape, '| sfreq:', sfreq)
print('  ', dict(zip(*np.unique(y_T, return_counts=True))))

print('Loading A01E (evaluation session)...')
X_E, y_E, _ = load_mat_runs(mat_path_E)
print('  ', X_E.shape)
print('  ', dict(zip(*np.unique(y_E, return_counts=True))))

X_all = np.concatenate([X_T, X_E], axis=0)
y_all = np.concatenate([y_T, y_E], axis=0)
print('Pooled:', X_all.shape, '|', dict(zip(*np.unique(y_all, return_counts=True))))

idx = np.arange(X_all.shape[0])
idx_train, idx_test, y_train, y_test = train_test_split(
    idx, y_all, test_size=0.2, stratify=y_all, random_state=42,
)

bands = {'mu': (8.0, 12.0), 'beta': (13.0, 30.0)}
for name, (lo, hi) in bands.items():
    b, a = butter(N=4, Wn=[lo, hi], btype='bandpass', fs=sfreq)
    Xf = filtfilt(b, a, X_all, axis=-1).astype(np.float32)
    np.save(os.path.join(CROSS_PROJECT_DIR, 'X_train_{}.npy'.format(name)), Xf[idx_train])
    np.save(os.path.join(CROSS_PROJECT_DIR, 'X_test_{}.npy'.format(name)), Xf[idx_test])
    print('{} band: train {} test {}'.format(name, Xf[idx_train].shape, Xf[idx_test].shape))

np.save(os.path.join(CROSS_PROJECT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(CROSS_PROJECT_DIR, 'y_test.npy'), y_test)
print('Train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using 17 channels: ['FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4']
Loading A01T (training session)...
   (288, 17, 1000) | sfreq: 250.0
   {np.str_('feet'): np.int64(72), np.str_('left_hand'): np.int64(72), np.str_('right_hand'): np.int64(72), np.str_('tongue'): np.int64(72)}
Loading A01E (evaluation session)...
   (288, 17, 1000)
   {np.str_('feet'): np.int64(72), np.str_('left_hand'): np.int64(72), np.str_('right_hand'): np.int64(72), np.str_('tongue'): np.int64(72)}
Pooled: (576, 17, 1000) | {np.str_('feet'): np.int64(144), np.str_('left_hand'): np.int64(144), np.str_('right_hand'): np.int64(144), np.str_('tongue'): np.int64(144)}
mu band: train (460, 17, 1000) test (116, 17, 1000)
beta band: train (460, 17, 1000) test (116, 17, 1000)
Train class counts: {np.str_('feet'): np.int64(115), np.st

In [ ]:
# CELL 3: FBCSP + QGA on BCIC-IV-2a (with dependency install)

!pip install -q mne scikit-learn joblib

import os
import numpy as np
import joblib
from mne.decoding import CSP
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'

BANDS = ['mu', 'beta']
N_COMP = 4

y_train = np.load(os.path.join(CROSS_PROJECT_DIR, 'y_train.npy'), allow_pickle=True)
y_test = np.load(os.path.join(CROSS_PROJECT_DIR, 'y_test.npy'), allow_pickle=True)

csp_models = {}
train_feats, test_feats = [], []
for band in BANDS:
    Xtr = np.load(os.path.join(CROSS_PROJECT_DIR, 'X_train_{}.npy'.format(band)))
    Xte = np.load(os.path.join(CROSS_PROJECT_DIR, 'X_test_{}.npy'.format(band)))
    csp = CSP(n_components=N_COMP, reg=None, log=True, norm_trace=False)
    ftr = csp.fit_transform(Xtr, y_train)
    fte = csp.transform(Xte)
    csp_models[band] = csp
    train_feats.append(ftr)
    test_feats.append(fte)
    print('{} CSP -> train {} test {}'.format(band, ftr.shape, fte.shape))

X_train_fbcsp = np.concatenate(train_feats, axis=1).astype(np.float32)
X_test_fbcsp = np.concatenate(test_feats, axis=1).astype(np.float32)
print('Concatenated FBCSP train:', X_train_fbcsp.shape)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_fbcsp).astype(np.float32)
X_test = scaler.transform(X_test_fbcsp).astype(np.float32)

joblib.dump(csp_models, os.path.join(CROSS_PROJECT_DIR, 'csp_models.joblib'))
joblib.dump(scaler, os.path.join(CROSS_PROJECT_DIR, 'fbcsp_scaler.joblib'))
np.save(os.path.join(CROSS_PROJECT_DIR, 'X_train_scaled.npy'), X_train)
np.save(os.path.join(CROSS_PROJECT_DIR, 'X_test_scaled.npy'), X_test)

# ---------------------------------------------------------------------------
# QGA feature selection
# ---------------------------------------------------------------------------
N_FEATURES = X_train.shape[1]
POP_SIZE = 20
N_GEN = 30
MIN_FEATURES = 3

RNG = np.random.default_rng(42)
DELTA_THETA = 0.05 * np.pi

alpha = np.full((POP_SIZE, N_FEATURES), 1.0 / np.sqrt(2.0))
beta = np.full((POP_SIZE, N_FEATURES), 1.0 / np.sqrt(2.0))

def measure(a_row, b_row, rng):
    prob_one = b_row ** 2
    return (rng.random(N_FEATURES) < prob_one).astype(np.int32)

def fitness(mask):
    if mask.sum() < MIN_FEATURES:
        return 0.0
    Xs = X_train[:, mask.astype(bool)]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = SVC(kernel='rbf', class_weight='balanced', random_state=42)
    return float(cross_val_score(clf, Xs, y_train, cv=cv, scoring='f1_macro', n_jobs=-1).mean())

def rotate(a_row, b_row, x_current, b_best, current_fit, best_fit):
    for j in range(N_FEATURES):
        if x_current[j] == b_best[j]:
            continue
        sign = 1.0 if b_best[j] == 1 else -1.0
        if current_fit > best_fit:
            sign = -sign
        theta = sign * DELTA_THETA
        a_new = a_row[j] * np.cos(theta) - b_row[j] * np.sin(theta)
        b_new = a_row[j] * np.sin(theta) + b_row[j] * np.cos(theta)
        a_row[j] = a_new
        b_row[j] = b_new

best_mask, best_fit = None, -1.0
for gen in range(N_GEN):
    masks = np.array([measure(alpha[i], beta[i], RNG) for i in range(POP_SIZE)])
    fits = np.array([fitness(m) for m in masks])
    idx_best = int(np.argmax(fits))
    if fits[idx_best] > best_fit:
        best_fit = float(fits[idx_best])
        best_mask = masks[idx_best].copy()
    for i in range(POP_SIZE):
        rotate(alpha[i], beta[i], masks[i], best_mask, fits[i], best_fit)
    print('Gen {:3d} | best CV F1 {:.4f} | mean {:.4f} | size {}'.format(
        gen + 1, best_fit, fits.mean(), int(best_mask.sum())))

print('Final selected indices:', np.where(best_mask == 1)[0].tolist())
print('Final best CV macro F1: {:.4f}'.format(best_fit))

np.save(os.path.join(CROSS_PROJECT_DIR, 'qga_best_mask.npy'), best_mask)
joblib.dump({'mask': best_mask, 'fitness': best_fit}, os.path.join(CROSS_PROJECT_DIR, 'qga_result.joblib'))
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Computing rank from data with rank=None
    Using tolerance 28 (2.2e-16 eps * 17 dim * 7.5e+15  max singular value)
    Estimated rank (data): 17
    data: rank 17 computed from 17 data channels with 0 projectors
Reducing data rank from 17 -> 17
Estimating class=feet covariance using EMPIRICAL
Done.
Estimating class=left_hand covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
Estimating class=tongue covariance using EMPIRICAL
Done.
mu CSP -> train (460, 4) test (116, 4)
Computing rank from data with rank=None
    Using tolerance 34 (2.2e-16 eps * 17 dim * 8.9e+15  max singular value)
    Estimated rank (data): 17
    data: rank 17 computed from 17 data channels with 0 projectors
Reducing data rank from 17 -> 17
Estimating class=feet covariance using EMPIRICAL
Done.
Estimating class=left_hand covariance using EMPIRICA

In [ ]:
# CELL 4: SVM on QGA-selected features + comparison + evaluation

import os
import numpy as np
import joblib
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'

X_train = np.load(os.path.join(CROSS_PROJECT_DIR, 'X_train_scaled.npy'))
X_test = np.load(os.path.join(CROSS_PROJECT_DIR, 'X_test_scaled.npy'))
y_train = np.load(os.path.join(CROSS_PROJECT_DIR, 'y_train.npy'), allow_pickle=True)
y_test = np.load(os.path.join(CROSS_PROJECT_DIR, 'y_test.npy'), allow_pickle=True)
mask = np.load(os.path.join(CROSS_PROJECT_DIR, 'qga_best_mask.npy')).astype(bool)

classes = ['left_hand', 'right_hand', 'feet', 'tongue']

# Baseline: majority class = 25% (4 balanced classes).
print('Baseline accuracy (chance): 0.2500')

param_grid = {'C': [0.1, 1.0, 10.0, 100.0, 1000.0],
              'gamma': ['scale', 0.01, 0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def fit_and_report(X_tr, X_te, tag):
    base = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    grid = GridSearchCV(base, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_tr, y_train)
    model = grid.best_estimator_
    y_pred = model.predict(X_te)
    print('--- {} ---'.format(tag))
    print('Best params:', grid.best_params_)
    print('CV f1_macro: {:.4f}'.format(grid.best_score_))
    print('Test accuracy: {:.4f}'.format(accuracy_score(y_test, y_pred)))
    print('Test macro F1: {:.4f}'.format(f1_score(y_test, y_pred, average='macro', zero_division=0)))
    print(classification_report(y_test, y_pred, labels=classes, digits=4, zero_division=0))
    print('Confusion matrix (rows=true, cols=pred):')
    print(confusion_matrix(y_test, y_pred, labels=classes))
    return model

full_model = fit_and_report(X_train, X_test, 'FULL FBCSP (8 feat) + SVM')
qga_model = fit_and_report(X_train[:, mask], X_test[:, mask],
                           'QGA-SELECTED ({} feat) + SVM'.format(int(mask.sum())))

joblib.dump(full_model, os.path.join(CROSS_PROJECT_DIR, 'svm_full.joblib'))
joblib.dump(qga_model, os.path.join(CROSS_PROJECT_DIR, 'svm_qga.joblib'))
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Baseline accuracy (chance): 0.2500
--- FULL FBCSP (8 feat) + SVM ---
Best params: {'C': 10.0, 'gamma': 0.01}
CV f1_macro: 0.5437
Test accuracy: 0.5000
Test macro F1: 0.4921
              precision    recall  f1-score   support

   left_hand     0.5000    0.4828    0.4912        29
  right_hand     0.5455    0.4138    0.4706        29
        feet     0.5000    0.3793    0.4314        29
      tongue     0.4773    0.7241    0.5753        29

    accuracy                         0.5000       116
   macro avg     0.5057    0.5000    0.4921       116
weighted avg     0.5057    0.5000    0.4921       116

Confusion matrix (rows=true, cols=pred):
[[14  7  3  5]
 [ 8 12  4  5]
 [ 3  2 11 13]
 [ 3  1  4 21]]
--- QGA-SELECTED (7 feat) + SVM ---
Best params: {'C': 1.0, 'gamma': 'scale'}
CV f1_macro: 0.5346
Test accuracy: 0.4483
Test macro F1: 0.4353
              preci

In [ ]:
# CELL C1: BCIC-IV-2a 2-class (left vs right), reload, filter, split, save

import os
import numpy as np
import scipy.io
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'

mat_path_T = os.path.join(CROSS_PROJECT_DIR, 'A01T.mat')
mat_path_E = os.path.join(CROSS_PROJECT_DIR, 'A01E.mat')
assert os.path.exists(mat_path_T), 'Missing A01T.mat'
assert os.path.exists(mat_path_E), 'Missing A01E.mat'

CH_NAMES = ['Fz','FC3','FC1','FCz','FC2','FC4','C5','C3','C1','Cz','C2','C4','C6',
            'CP3','CP1','CPz','CP2','CP4','P1','Pz','P2','POz']
COMMON_CHANNELS = ['FC3','FC1','FCz','FC2','FC4','C5','C3','C1','Cz','C2','C4','C6',
                   'CP3','CP1','CPz','CP2','CP4']
ch_idx = [CH_NAMES.index(c) for c in COMMON_CHANNELS]

# 2-class filter: keep only left_hand (label 1) and right_hand (label 2).
LABEL_MAP = {1: 'left_hand', 2: 'right_hand', 3: 'feet', 4: 'tongue'}
KEEP_LABELS = {1, 2}

def load_mat_runs_2class(path):
    mat = scipy.io.loadmat(path, squeeze_me=True, struct_as_record=False)
    data = mat['data']
    all_trials, all_labels = [], []
    sfreq = None
    for run in data:
        X = run.X
        trials = run.trial
        y = run.y
        if sfreq is None:
            sfreq = float(run.fs)
        epoch_len = int(sfreq * 4.0)
        for onset, lbl in zip(trials, y):
            lbl = int(lbl)
            if lbl not in KEEP_LABELS:
                continue
            start = int(onset)
            end = start + epoch_len
            if end > X.shape[0]:
                continue
            epoch = X[start:end, ch_idx].T
            all_trials.append(epoch)
            all_labels.append(LABEL_MAP[lbl])
    return np.stack(all_trials).astype(np.float32), np.array(all_labels), sfreq

X_T, y_T, sfreq = load_mat_runs_2class(mat_path_T)
X_E, y_E, _ = load_mat_runs_2class(mat_path_E)
print('T session (2-class):', X_T.shape, dict(zip(*np.unique(y_T, return_counts=True))))
print('E session (2-class):', X_E.shape, dict(zip(*np.unique(y_E, return_counts=True))))

X_all = np.concatenate([X_T, X_E], axis=0)
y_all = np.concatenate([y_T, y_E], axis=0)
print('Pooled (2-class):', X_all.shape, dict(zip(*np.unique(y_all, return_counts=True))))

idx = np.arange(X_all.shape[0])
idx_train, idx_test, y_train, y_test = train_test_split(
    idx, y_all, test_size=0.2, stratify=y_all, random_state=42,
)

bands = {'mu': (8.0, 12.0), 'beta': (13.0, 30.0)}
for name, (lo, hi) in bands.items():
    b, a = butter(N=4, Wn=[lo, hi], btype='bandpass', fs=sfreq)
    Xf = filtfilt(b, a, X_all, axis=-1).astype(np.float32)
    np.save(os.path.join(CROSS_PROJECT_DIR, '2c_X_train_{}.npy'.format(name)), Xf[idx_train])
    np.save(os.path.join(CROSS_PROJECT_DIR, '2c_X_test_{}.npy'.format(name)), Xf[idx_test])
    print('{} band: train {} test {}'.format(name, Xf[idx_train].shape, Xf[idx_test].shape))

np.save(os.path.join(CROSS_PROJECT_DIR, '2c_y_train.npy'), y_train)
np.save(os.path.join(CROSS_PROJECT_DIR, '2c_y_test.npy'), y_test)
print('Train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
T session (2-class): (144, 17, 1000) {np.str_('left_hand'): np.int64(72), np.str_('right_hand'): np.int64(72)}
E session (2-class): (144, 17, 1000) {np.str_('left_hand'): np.int64(72), np.str_('right_hand'): np.int64(72)}
Pooled (2-class): (288, 17, 1000) {np.str_('left_hand'): np.int64(144), np.str_('right_hand'): np.int64(144)}
mu band: train (230, 17, 1000) test (58, 17, 1000)
beta band: train (230, 17, 1000) test (58, 17, 1000)
Train class counts: {np.str_('left_hand'): np.int64(115), np.str_('right_hand'): np.int64(115)}
Test class counts: {np.str_('left_hand'): np.int64(29), np.str_('right_hand'): np.int64(29)}
Saved to: /content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/


In [ ]:
# CELL C2: FBCSP + QGA + SVM on 2-class BCIC-IV-2a

!pip install -q mne scikit-learn joblib

import os
import numpy as np
import joblib
from mne.decoding import CSP
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'

BANDS = ['mu', 'beta']
N_COMP = 4

y_train = np.load(os.path.join(CROSS_PROJECT_DIR, '2c_y_train.npy'), allow_pickle=True)
y_test = np.load(os.path.join(CROSS_PROJECT_DIR, '2c_y_test.npy'), allow_pickle=True)

# FBCSP extraction.
csp_models = {}
train_feats, test_feats = [], []
for band in BANDS:
    Xtr = np.load(os.path.join(CROSS_PROJECT_DIR, '2c_X_train_{}.npy'.format(band)))
    Xte = np.load(os.path.join(CROSS_PROJECT_DIR, '2c_X_test_{}.npy'.format(band)))
    csp = CSP(n_components=N_COMP, reg=None, log=True, norm_trace=False)
    ftr = csp.fit_transform(Xtr, y_train)
    fte = csp.transform(Xte)
    csp_models[band] = csp
    train_feats.append(ftr)
    test_feats.append(fte)
    print('{} CSP -> train {} test {}'.format(band, ftr.shape, fte.shape))

X_train_fbcsp = np.concatenate(train_feats, axis=1).astype(np.float32)
X_test_fbcsp = np.concatenate(test_feats, axis=1).astype(np.float32)
print('Concatenated FBCSP train:', X_train_fbcsp.shape)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_fbcsp).astype(np.float32)
X_test = scaler.transform(X_test_fbcsp).astype(np.float32)

# ---------------------------------------------------------------------------
# QGA
# ---------------------------------------------------------------------------
N_FEATURES = X_train.shape[1]
POP_SIZE = 20
N_GEN = 30
MIN_FEATURES = 3
RNG = np.random.default_rng(42)
DELTA_THETA = 0.05 * np.pi

alpha = np.full((POP_SIZE, N_FEATURES), 1.0 / np.sqrt(2.0))
beta = np.full((POP_SIZE, N_FEATURES), 1.0 / np.sqrt(2.0))

def measure(a_row, b_row, rng):
    prob_one = b_row ** 2
    return (rng.random(N_FEATURES) < prob_one).astype(np.int32)

def fitness(mask):
    if mask.sum() < MIN_FEATURES:
        return 0.0
    Xs = X_train[:, mask.astype(bool)]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = SVC(kernel='rbf', class_weight='balanced', random_state=42)
    return float(cross_val_score(clf, Xs, y_train, cv=cv, scoring='f1_macro', n_jobs=-1).mean())

def rotate(a_row, b_row, x_current, b_best, current_fit, best_fit):
    for j in range(N_FEATURES):
        if x_current[j] == b_best[j]:
            continue
        sign = 1.0 if b_best[j] == 1 else -1.0
        if current_fit > best_fit:
            sign = -sign
        theta = sign * DELTA_THETA
        a_new = a_row[j] * np.cos(theta) - b_row[j] * np.sin(theta)
        b_new = a_row[j] * np.sin(theta) + b_row[j] * np.cos(theta)
        a_row[j] = a_new
        b_row[j] = b_new

best_mask, best_fit = None, -1.0
for gen in range(N_GEN):
    masks = np.array([measure(alpha[i], beta[i], RNG) for i in range(POP_SIZE)])
    fits = np.array([fitness(m) for m in masks])
    idx_best = int(np.argmax(fits))
    if fits[idx_best] > best_fit:
        best_fit = float(fits[idx_best])
        best_mask = masks[idx_best].copy()
    for i in range(POP_SIZE):
        rotate(alpha[i], beta[i], masks[i], best_mask, fits[i], best_fit)
    print('Gen {:3d} | best CV F1 {:.4f} | mean {:.4f} | size {}'.format(
        gen + 1, best_fit, fits.mean(), int(best_mask.sum())))

print('Final selected indices:', np.where(best_mask == 1)[0].tolist())
print('Final best CV macro F1: {:.4f}'.format(best_fit))

# ---------------------------------------------------------------------------
# SVM eval on full and QGA subsets
# ---------------------------------------------------------------------------
classes = ['left_hand', 'right_hand']
print('Baseline accuracy (chance): 0.5000')

param_grid = {'C': [0.1, 1.0, 10.0, 100.0, 1000.0],
              'gamma': ['scale', 0.01, 0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def fit_and_report(X_tr, X_te, tag):
    base = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    grid = GridSearchCV(base, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_tr, y_train)
    model = grid.best_estimator_
    y_pred = model.predict(X_te)
    print('--- {} ---'.format(tag))
    print('Best params:', grid.best_params_)
    print('CV f1_macro: {:.4f}'.format(grid.best_score_))
    print('Test accuracy: {:.4f}'.format(accuracy_score(y_test, y_pred)))
    print('Test macro F1: {:.4f}'.format(f1_score(y_test, y_pred, average='macro', zero_division=0)))
    print(classification_report(y_test, y_pred, labels=classes, digits=4, zero_division=0))
    print('Confusion matrix (rows=true, cols=pred):')
    print(confusion_matrix(y_test, y_pred, labels=classes))
    return model

full_model = fit_and_report(X_train, X_test, 'FULL FBCSP (8 feat) + SVM')
qga_model = fit_and_report(X_train[:, best_mask.astype(bool)], X_test[:, best_mask.astype(bool)],
                           'QGA-SELECTED ({} feat) + SVM'.format(int(best_mask.sum())))

joblib.dump(full_model, os.path.join(CROSS_PROJECT_DIR, '2c_svm_full.joblib'))
joblib.dump(qga_model, os.path.join(CROSS_PROJECT_DIR, '2c_svm_qga.joblib'))
np.save(os.path.join(CROSS_PROJECT_DIR, '2c_qga_best_mask.npy'), best_mask)
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Computing rank from data with rank=None
    Using tolerance 20 (2.2e-16 eps * 17 dim * 5.3e+15  max singular value)
    Estimated rank (data): 17
    data: rank 17 computed from 17 data channels with 0 projectors
Reducing data rank from 17 -> 17
Estimating class=left_hand covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
mu CSP -> train (230, 4) test (58, 4)
Computing rank from data with rank=None
    Using tolerance 23 (2.2e-16 eps * 17 dim * 6.2e+15  max singular value)
    Estimated rank (data): 17
    data: rank 17 computed from 17 data channels with 0 projectors
Reducing data rank from 17 -> 17
Estimating class=left_hand covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
beta CSP -> train (230, 4) test (58, 4)
Concatenated FBCSP train: (230, 8)
Gen   1 | best CV F1 0.7